In [23]:
import sklearn
import argparse
import mlflow
from mlflow.models import infer_signature

from mlflow import MlflowClient
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [36]:
model_name = "iris_model"

# parser = argparse.ArgumentParser()
# parser.add_argument("--n-estimators",type=int,default=1,help="Number of estimators for the random forest model")
# parser.add_argument("--max-depth",type=int,default=5,help="Maximum depth of the random forest model")
# args = parser.parse_args()

# n_estimators = args.n_estimators
# max_depth = args.max_depth


n_estimators = 1
max_depth = 5

In [ ]:
iris_data = load_iris()

In [37]:
X_train,X_test,y_train,y_test = train_test_split(iris_data['data'],iris_data['target'],test_size=0.2,random_state=42)



In [38]:
mlflow.set_tracking_uri("http://localhost:5002")


In [ ]:
import subprocess, sys, time

log = open("mlflow_server.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "mlflow", "server",
     "--host", "127.0.0.1", "--port", "5001",
     "--backend-store-uri", "sqlite:///mlflow.db"],
    stdout=log, stderr=subprocess.STDOUT,
)

time.sleep(10)



mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("iris_experiment")

with mlflow.start_run() as run:
    model = RandomForestClassifier(n_estimators=n_estimators,max_depth=max_depth)
    model.fit(X_train,y_train)

    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test,preds)
    f1 = f1_score(y_test,preds,average='weighted')
    precision = precision_score(y_test,preds,average='weighted')
    recall = recall_score(y_test,preds,average='weighted')

    mlflow.log_params({"n_estimators": n_estimators, "max_depth": max_depth})
    mlflow.log_metrics({"accuracy": accuracy, "f1": f1, "precision": precision, "recall": recall})


    info = mlflow.sklearn.log_model(sk_model=model,
    name="iter1",
    signature=infer_signature(X_train,model.predict(X_train)),
    registered_model_name=model_name,
    skops_trusted_types=["sklearn.tree._tree.Tree"])
    
    version = info.registered_model_version

    print (f"model registered with run_id:{run.info.run_id} and version: {version}")






Registered model 'iris_model' already exists. Creating a new version of this model...
2026/10/03 16:09:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris_model, version 3


🏃 View run painted-dog-313 at: http://localhost:5001/#/experiments/1/runs/5d50853f723c44a39b03588a4971a7fe
🧪 View experiment at: http://localhost:5001/#/experiments/1


Created version '3' of model 'iris_model'.


: 